# AI-Based Emergency Blood Donor Recommendation System
## Academic ML Model Training, Evaluation, and Benchmarking Notebook

**Author**: AI Healthcare Decision-Support Lab  
**Course**: Sem 4 AI Group Project  
**Objective**: Benchmark Machine Learning classifiers (*Random Forest, Logistic Regression, Decision Tree*) to predict donor affirmative emergency response likelihood ($P(\text{Response} = 1)$).

> **Academic Disclaimer**: Dataset contains synthetic/demo donor records created for academic research purposes. Final clinical compatibility and donor eligibility decisions must always be verified by certified healthcare professionals.

### 1. Environment Setup and Data Loading

In [ ]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is available
sys.path.insert(0, os.path.abspath(".."))

from src.preprocessing import ML_FEATURE_NAMES, calculate_distance
from src.data_generator import generate_synthetic_donors, save_and_init_db
from src.train_model import engineer_training_features, train_and_evaluate_models

# Set style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

# Load dataset
data_path = os.path.join("..", "data", "donors.csv")
if not os.path.exists(data_path):
    df, _ = save_and_init_db(data_dir=os.path.join("..", "data"), db_path=os.path.join("..", "data", "blood_donor.db"))
else:
    df = pd.read_csv(data_path)

print(f"Loaded {len(df)} synthetic donor records.")
df.head()

### 2. Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics
print("Dataset Information:")
print(df.info())

print("\nDistribution of Target Response:")
print(df["target_response"].value_counts(normalize=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Blood Group Distribution
sns.countplot(data=df, x="blood_group", order=df["blood_group"].value_counts().index, ax=axes[0], palette="Reds_r")
axes[0].set_title("Distribution of Donor Blood Groups")
axes[0].set_xlabel("Blood Group")
axes[0].set_ylabel("Count")

# Target Response vs Availability
sns.countplot(data=df, x="available", hue="target_response", ax=axes[1], palette="coolwarm")
axes[1].set_title("Target Response by Current Availability")
axes[1].set_xlabel("Available (1 = Yes, 0 = No)")
axes[1].set_ylabel("Count")
axes[1].legend(title="Target Response", labels=["Unlikely (0)", "Likely (1)"])

plt.tight_layout()
plt.show()

### 3. Feature Engineering and Model Benchmarking

In [ ]:
# Prepare features
X, y = engineer_training_features(df)
print(f"Feature matrix shape: {X.shape}, Target shape: {y.shape}")
print(f"Features utilized: {ML_FEATURE_NAMES}")

# Execute model training and benchmarking
results, best_model = train_and_evaluate_models(X, y, random_state=42)

### 4. Comparative Evaluation Metrics

In [ ]:
metrics_comparison = []
for model_name in ["Random Forest", "Logistic Regression", "Decision Tree"]:
    m = results[model_name]
    metrics_comparison.append({
        "Model": model_name,
        "Accuracy": m["accuracy"],
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1 Score": m["f1_score"],
        "ROC-AUC": m["roc_auc"],
        "5-Fold CV F1 (Mean ± Std)": f"{m['cv_f1_mean']} ± {m['cv_f1_std']}"
    })

metrics_df = pd.DataFrame(metrics_comparison)
display(metrics_df)

### 5. Confusion Matrices Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, model_name in enumerate(["Random Forest", "Logistic Regression", "Decision Tree"]):
    cm = np.array(results[model_name]["confusion_matrix"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[idx],
                xticklabels=["Unlikely (0)", "Likely (1)"],
                yticklabels=["Unlikely (0)", "Likely (1)"])
    axes[idx].set_title(f"{model_name} Confusion Matrix")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")

plt.tight_layout()
plt.show()

### 6. Random Forest Feature Importance Analysis

*Note*: Feature importances indicate statistical contribution to the decision forest during training; they do not imply direct clinical causality.

In [ ]:
feat_imp = results["feature_importances"]
feat_df = pd.DataFrame(list(feat_imp.items()), columns=["Feature", "Importance"]).sort_values(by="Importance", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feat_df["Feature"], feat_df["Importance"], color="#c0392b")
plt.title("Random Forest Feature Importance Analysis", fontsize=14, fontweight="bold")
plt.xlabel("Relative Importance (Gini Impurity Reduction)")
plt.ylabel("Feature")
for i, v in enumerate(feat_df["Importance"]):
    plt.text(v + 0.005, i, f"{v:.4f}", va="center", fontweight="bold")
plt.xlim(0, max(feat_df["Importance"]) + 0.05)
plt.tight_layout()
plt.show()